**Recorded scaffold run: NOT COMPLETED.** This fresh run saved 3 expected `NotImplementedError` outputs. These are unfinished exercises, not passes. Read the paper questions first, implement Tier 2, then return for code checks.

**STUDENT EXERCISES — INCOMPLETE.** These notebooks intentionally call unimplemented functions. TODO errors are expected until the learner implements `student_impl.py`. Instructor solutions are outside this folder.

# Tier2_Student — implement and challenge your model

Budget: 2–3 hours after Tier 1. Work in `student_impl.py`. Run `python -m pytest -q tests` from this directory. Passing the notebook alone does not show the functions work for every required case.

Published view: expected TODO error names/messages are preserved; machine-specific traceback paths are omitted.

In [1]:
from pathlib import Path
import math
import torch
import torch.nn.functional as F
import student_impl as work

torch.manual_seed(9)
torch.set_num_threads(1)
print('Python implementation:', Path(work.__file__).name)
print('Torch:', torch.__version__)


Python implementation: student_impl.py
Torch: 2.10.0+cpu


## 2A — scaled dot-product attention (8 points)
Implement `attention`: derive QK^T / sqrt(d), apply a floating additive mask, normalize over keys, then combine V. Support rectangular attention and broadcast masks. Chosen edge policy: a query with no valid keys has zero weights/output. Explain why the usual row-sum=1 claim has this exception. Boolean masks are outside this interface because APIs use different conventions.

In [2]:
q=torch.randn(2,3,4,8);k=torch.randn(2,3,7,8);v=torch.randn(2,3,7,5)
actual,weights=work.attention(q,k,v)
expected=F.scaled_dot_product_attention(q,k,v,dropout_p=0.)
torch.testing.assert_close(actual,expected,atol=2e-6,rtol=2e-6)
print('Independent PyTorch reference agrees.')

NotImplementedError: Implement scaled dot-product attention (Tier 2A).

## 2B — test causal behavior, not only mask appearance (7 points)
Implement `causal_mask`; the query at local index i can attend key j iff j<=past+i. Change only future keys/values and verify earlier outputs do not change. Also test an all-masked row and a padding-key mask with `tests/test_student_impl.py`. State which tests would fail if the triangular direction were reversed.

In [3]:
q=torch.randn(1,1,4,8);k=torch.randn(1,1,4,8);v=torch.randn(1,1,4,6)
mask=work.causal_mask(4)
original,_=work.attention(q,k,v,mask)
changed_v=v.clone();changed_v[:,:,2:]=100
changed,_=work.attention(q,k,changed_v,mask)
torch.testing.assert_close(original[:,:,:2],changed[:,:,:2])
all_blocked=torch.full((4,4),float('-inf'))
zero,weights=work.attention(q,k,v,all_blocked)
assert zero.abs().sum()==0 and weights.abs().sum()==0
print('Future invariance and all-masked policy verified.')

NotImplementedError: Implement an additive causal mask (Tier 2B).

## 2C — target shift and padding-aware loss (7 points)
Implement `next_token_pairs` and `masked_cross_entropy`. The latter averages valid labels only; use an explicit ignore index. Chosen policy: all-ignored labels yield differentiable zero. An alternative is to reject/skip that batch, but never train on an accidental NaN.

Ignoring PAD in CE removes those targets from loss. It does not mask padding keys in attention. Explain this distinction and describe both masks needed for variable-length source/target batches.

In [4]:
logits=torch.randn(1,3,7,requires_grad=True)
labels=torch.tensor([[0,6,2]])
loss=work.masked_cross_entropy(logits,labels,ignore_index=6)
expected=F.cross_entropy(logits[:,[0,2]].reshape(-1,7),torch.tensor([0,2]))
torch.testing.assert_close(loss,expected)
loss.backward()
assert logits.grad[:,1].abs().sum()==0
print('Loss:',float(loss.detach()),'padding gradient:',logits.grad[:,1])

NotImplementedError: Implement padding-aware target loss (Tier 2C).

## 2D — explain a failed design (3 points)
At decoder input index 1, teacher forcing already contains the label predicted at index 0. Explain how unmasked attention can copy it and why inference cannot. Use your future-invariance check to distinguish a correct masked implementation from a deliberately unmasked one.

## Readiness check
Submit your own implementation, test output, shape annotations and a 250-word explanation of why mask, shift, source attention and loss padding have distinct roles. For a next project, start with a simple baseline, overfit one tiny batch, use an independent split, inspect failed predictions, and report a limitation before adding complexity.